This notebook calculates the VaR and ES values for the historical, variance-covariance and monte-carlo method.




#Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import sys
sys.path.insert(0,'/content/drive/MyDrive/Masterthesis')
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats

Mounted at /content/drive


#Load Historical Data


In [ ]:
path_bluechip = "/content/drive/MyDrive/Masterthesis/Data/Datasets/bluechip_sp500.csv"
path_crossAsset = "/content/drive/MyDrive/Masterthesis/Data/Datasets/crossAsset_portfolio.csv"

df_bluechip = pd.read_csv(path_bluechip)
df_crossAsset = pd.read_csv(path_crossAsset)

#This method calculates the portfolio returns based on some weights
def portfolio_returns(returns, weights=None):
    returns = np.asarray(returns, dtype=float)
    returns = np.nan_to_num(returns)

    T, N = returns.shape
    if weights is None:
        weights = np.ones(N) / N
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / np.sum(weights)

    port_ret = returns @ weights
    return port_ret

returns_bluechip = portfolio_returns(df_bluechip)#[:-4]
returns_crossAsset = portfolio_returns(df_crossAsset)#[:-4]

In [ ]:
#this method calculates the actual VaR and ES values based on the method
def analyze_portfolio_risk(log_returns, start_index, window_size=250, n_sims=10000):

    alpha_var = 0.025   # 95% oder 97.5%
    alpha_es = 0.025   # 97.5%

    n = len(log_returns)
    results = {
        'VaR_Hist': np.full(n, np.nan), 'ES_Hist': np.full(n, np.nan),
        'VaR_VC': np.full(n, np.nan),   'ES_VC': np.full(n, np.nan),
        'VaR_MC': np.full(n, np.nan),   'ES_MC': np.full(n, np.nan)
    }




    for i in range(start_index, n):
        # the observed window is (i - window_size) to i
        window_data = log_returns[i-window_size:i]

        # 1. Historical method
        var_hist = np.percentile(window_data, alpha_var * 100)
        es_thresh_hist = np.percentile(window_data, alpha_es * 100)
        es_hist = window_data[window_data <= es_thresh_hist].mean()

        results['VaR_Hist'][i] = var_hist
        results['ES_Hist'][i] = es_hist

        # 2. Variance-Covariance
        mu = np.mean(window_data)
        sigma = np.std(window_data)

        results['VaR_VC'][i] = mu + sigma * stats.norm.ppf(alpha_var)
        z_es = stats.norm.ppf(alpha_es)
        results['ES_VC'][i] = mu - (stats.norm.pdf(z_es) / alpha_es) * sigma

        # 3. Monte Carlo
        simulated_returns = np.random.normal(mu, sigma, n_sims)
        results['VaR_MC'][i] = np.percentile(simulated_returns, alpha_var * 100)
        es_thresh_mc = np.percentile(simulated_returns, alpha_es * 100)
        results['ES_MC'][i] = simulated_returns[simulated_returns <= es_thresh_mc].mean()


    plot_indices = np.arange(start_index, n)


    relevant_returns = log_returns[start_index:]
    cumulative_returns = np.cumsum(relevant_returns)
    portfolio_values = 100 * np.exp(cumulative_returns)
    #plotting values
    portfolio_plot_values = np.insert(portfolio_values, 0, 100)
    portfolio_plot_indices = np.insert(plot_indices, 0, start_index)

    # relevent slices for the results
    res_sliced = {k: v[start_index:] for k, v in results.items()}
    returns_sliced = log_returns[start_index:]


    fig, axes = plt.subplots(4, 1, figsize=(12, 24), sharex=True)

    # Plot 1: Portfolio Value
    axes[0].plot(portfolio_plot_indices, portfolio_plot_values, color='blue', lw=2)
    axes[0].set_title(f'Portfolio Entwicklung (Indexiert auf 100 ab t={start_index})')
    axes[0].set_ylabel('Index Wert')
    axes[0].grid(True, alpha=0.3)

    # Loop 3 methods
    methods = [
        ("Historische Simulation", "Hist"),
        ("Varianz-Kovarianz (VC)", "VC"),
        ("Monte Carlo (MC)", "MC")
    ]

    for idx, (method_name, suffix) in enumerate(methods):
        ax = axes[idx + 1]

        var_series = res_sliced[f'VaR_{suffix}']
        es_series = res_sliced[f'ES_{suffix}']

        # Returns
        ax.plot(plot_indices, returns_sliced, color='grey', alpha=0.4, label='Log Returns', lw=1)

        # VaR & ES
        ax.plot(plot_indices, var_series, color='purple', label=f'VaR 95%', lw=1.5)
        ax.plot(plot_indices, es_series, color='red', linestyle='--', label=f'ES 97.5%', lw=1.5)

        # Exceedances
        exceedances_mask = returns_sliced < var_series
        ax.scatter(plot_indices[exceedances_mask], returns_sliced[exceedances_mask],
                   color='black', marker='x', s=40, label='Exceedance', zorder=5)

        ax.set_title(f'Methode: {method_name}')
        ax.set_ylabel('Log Return')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

    plt.xlabel('Zeit (Index)')
    plt.tight_layout()
    plt.show()

    return results, res_sliced # Return both full results and sliced results

In [ ]:
#this method safes the results with the correct names later used in the R files
def save_risk_metrics_to_csv(results_dict, base_file_path):
    methods = [
        ("Historisch", "historical"),
        ("Varianz-Kovarianz", "variance_covariance"),
        ("Monte Carlo", "monte_carlo")
    ]

    for method_name, suffix in methods:
        if suffix == 'historical':
            df_method_results = pd.DataFrame({
                'VaR': results_dict['VaR_Hist'],
                'ES': results_dict['ES_Hist']
            })
        elif suffix == 'variance_covariance':
            df_method_results = pd.DataFrame({
                'VaR': results_dict['VaR_VC'],
                'ES': results_dict['ES_VC']
            })
        elif suffix == 'monte_carlo':
            df_method_results = pd.DataFrame({
                'VaR': results_dict['VaR_MC'],
                'ES': results_dict['ES_MC']
            })


        file_name_parts = list(os.path.splitext(base_file_path))
        method_file_path = f"{file_name_parts[0]}_{suffix}{file_name_parts[1]}.csv"

        df_method_results.to_csv(method_file_path, index=False)

In [ ]:
#risk_results_full, risk_results_sliced = analyze_portfolio_risk(returns_bluechip,start_index=4564, window_size=252, n_sims=1000)
risk_results_full, risk_results_sliced = analyze_portfolio_risk(returns_crossAsset,start_index=5298, window_size=252, n_sims=1000)


output_csv_path = '/content/drive/MyDrive/Masterthesis/Data/VaR_and_ES/crossAsset'
save_risk_metrics_to_csv(risk_results_sliced, output_csv_path)